In [2]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import featureman.gen_data as man
from sklearn.cluster import SpectralClustering
import pickle
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### what to do next
- decoder/dict_atoms -> sae_hidden x mlp_dim
- relu_acts -> sae_hidden
- out -> mlp_dim

- concat(samples X filtered_sae_hidden) x mlp_dim
- einop('s f m -> (s f) m')
- PCA of that stuff

In [3]:
model_dict = torch.load("modular_arithmetic_model.pth", map_location=device)
model = man.OneLayerTransformer(p=113, d_model=128, nheads=4).to(device)
model.load_state_dict(model_dict)

<All keys matched successfully>

In [4]:
torch.manual_seed(1337)
# generate combination of all inputs a and b range (113)
a = np.arange(113)
b = np.arange(113)
# generate inputs for the model
inputs = np.array([[a_i, 113, b_i, 114] for a_i in a for b_i in b])
inputs = torch.tensor(inputs).to(device)  # Add batch dimension
print(inputs.shape)
logits, activations = model(inputs, return_activations=True)
activations_data = activations[:, -1, :].detach()
batched_acts = activations_data.unsqueeze(0).repeat(5, 1, 1).to(device)
del model, model_dict
print(batched_acts.shape)

torch.Size([12769, 4])
torch.Size([5, 12769, 512])


In [5]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import numpy as np

reconstructions = batched_acts[3]

pca = PCA()
output_pca = pca.fit_transform(reconstructions.detach().cpu().numpy())

# Create single 3D plot
fig = go.Figure()

for a in range(20):
    fig.add_trace(
        go.Scatter3d(
            x=output_pca[:, 2][a*113:a*113+113],
            y=output_pca[:, 5][a*113:a*113+113],
            z=output_pca[:, 1][a*113:a*113+113],
            mode='markers',
            marker=dict(
                size=2,
                colorscale='viridis',
                opacity=0.3,
                symbol='x'
            ),
            name=f'Batch {a}'
        )
    )

# Update scene properties
fig.update_layout(
    scene=dict(
        xaxis_title="Principal Component 0",
        yaxis_title="Principal Component 3",
        zaxis_title="Principal Component 1"
    ),
    title="PCA Visualization of Reconstructions",
    width=800,
    height=700
)

fig.show()

In [22]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import numpy as np

reconstructions = batched_acts[3]

pca = PCA()
output_pca = pca.fit_transform(reconstructions.detach().cpu().numpy())

a_values = np.arange(113)
b_values = np.arange(113)

inputs = np.array([[a_i, 113, b_i, 114] for a_i in a_values for b_i in b_values])

targets = [
    50,
    51,
    52,
]

# Create single 3D plot
fig = go.Figure()

for i in targets:
    indices = np.where(inputs[:, 0] + inputs[:, 2] == i)[0]
    fig.add_trace(
        go.Scatter3d(
            x=output_pca[:, 0][indices],
            y=output_pca[:, 3][indices],
            z=output_pca[:, 1][indices],
            mode='markers',
            marker=dict(
                size=5,
                colorscale='viridis',
                opacity=0.6,
                symbol='x'
            ),
            name=f'Target {i}'
        )
    )

# Update scene properties
fig.update_layout(
    scene=dict(
        xaxis_title="Principal Component 0",
        yaxis_title="Principal Component 3",
        zaxis_title="Principal Component 1"
    ),
    title="PCA Visualization of Reconstructions",
    width=800,
    height=700
)

fig.show()